In [ ]:
# 데이터셋 : https://www.kaggle.com/code/lucamassaron/fine-tune-gemma-3-1b-it-for-sentiment-analysis/input

In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

import torch
import torch.nn as nn

import transformers
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments, # Note: SFTConfig from TRL is used later
                          pipeline,
                          logging)

# Explicitly import Gemma3ForCausalLM
from transformers.models.gemma3 import Gemma3ForCausalLM

from datasets import Dataset
from peft import LoraConfig, PeftConfig, PeftModel
from trl import SFTTrainer, SFTConfig # Use SFTConfig from TRL
import bitsandbytes as bnb

from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix)

from sklearn.model_selection import train_test_split

In [2]:
def define_device():
    """Determine and return the optimal PyTorch device based on availability."""

    print(f"PyTorch version: {torch.__version__}", end=" -- ")

    # Check if MPS (Metal Performance Shaders) is available for macOS
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("using MPS device on macOS")
        return torch.device("mps")

    # Check for CUDA availability
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"using {device}")
    return device

In [3]:
print(type(define_device()))

PyTorch version: 2.7.0+cu118 -- using cuda
<class 'torch.device'>


In [ ]:
# Determine optimal computation dtype based on GPU capability
# Use bfloat16 if Compute Capability >= 8.0, otherwise float16
compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f"Using compute dtype {compute_dtype}")

# Select the best available device (CPU, CUDA, or MPS)
device = define_device()
print(f"Operating on {device}")

Using compute dtype torch.bfloat16
PyTorch version: 2.7.0+cu118 -- using cuda
Operating on cuda


In [5]:
model = Gemma3ForCausalLM.from_pretrained(
        "google/gemma-3-1b-it",
        attn_implementation="eager",
        low_cpu_mem_usage=True,      # Reduces CPU RAM usage during loading
        device_map=device, 
    )

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [2]:
model.dtype

NameError: name 'model' is not defined

In [ ]:
model.config

Gemma3TextConfig {
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "eos_token_id": [
    1,
    106
  ],
  "final_logit_softcapping": null,
  "head_dim": 256,
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 1152,
  "initializer_range": 0.02,
  "intermediate_size": 6912,
  "max_position_embeddings": 32768,
  "model_type": "gemma3_text",
  "num_attention_heads": 4,
  "num_hidden_layers": 26,
  "num_key_value_heads": 1,
  "pad_token_id": 0,
  "query_pre_attn_scalar": 256,
  "rms_norm_eps": 1e-06,
  "rope_local_base_freq": 10000,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "sliding_window": 512,
  "sliding_window_pattern": 6,
  "torch_dtype": "float32",
  "transformers_version": "4.52.4",
  "use_cache": true,
  "vocab_size": 262144
}

In [7]:
model.device

device(type='cuda', index=0)

In [8]:
model.loss_type

'ForCausalLM'

In [9]:
model.vocab_size

262144

In [60]:
max_seq_length = 8192

tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-1b-it",
    max_seq_length=max_seq_length,
    device_map=device
)

EOS_TOKEN = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [61]:
EOS_TOKEN 

'<eos>'

In [ ]:
# 아직 쓰이지 않는 토큰 (reserved token placeholder)
# SentencePiece, BPE 계열에서 vocab_size를 크게 잡았는데 실제로 다 채우지 못하고 남을 때 발생
# 학습에는 거의 등장하지 않음 -> 임베딩이 랜덤 초기화상태
# 나중에 도메인 특화 토큰이나 새로운 특수 기호를 넣을 떄 활용
tokenizer.vocab['<unused5570>']

261472

In [66]:
tokenizer.vocab["▁때문"]

35168

In [69]:
print(list(tokenizer.vocab.keys())[:100])

['▁wah', '▁radium', '▁ناش', '▁赵', '▁अय', '儸', '▁représent', '▁स्वरुप', '-%', '▁publics', '最後まで', '<unused5570>', 'Phones', '酴', '▁πλα', '▁|\\', 'hame', '▁Codes', 'حدث', 'შირ', 'icen', '▁Pau', '▁immagin', '▁Kiz', '▁ocorr', '▁جلو', 'ppe', '▁recuperar', '▁embroiled', '▁ज्योग्राफी', '▁unim', '▁भिखारी', '２', '▁kantor', '▁Deutschlands', '▁cerevisiae', 'ونها', '▁IPs', 'ᥣ', 'দর্শনে', '▁documento', '▁때문', '▁Polytechn', '▁arbeit', 'lda', 'мены', 'আম', '▁REVIEW', '▁урока', '的企业', '▁எல்லாம்', '▁chce', 'suit', '▁vhodné', '▁राजू', '▁DDT', '<unused160>', '\uf0ad', '分からない', '▁trunks', '🤭', 'भूमि', 'Til', '▁IRS', '🈶', '▁indoors', 'witter', '▁tests', '▁tiem', 'startsWith', 'getInputStream', '▁ස්', '▁Sabbath', '▁burn', '▁ryzy', '婴儿', '▁আওয়ামী', 'lays', 'ș', '▁تخلی', '>-->', '▁usuarios', '▁Kiss', '▁الرئيسي', 'ulare', 'compact', 'opedia', 'ভোগ', '▁بزر', '▁Personen', '壟', 'inosa', '▁belles', '▁sounding', '▁hygienic', '▁Qh', '▁চাহিদা', 'brochen', 'が多く', '“…']


### Dataset

In [ ]:
file_path = '/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/5_sft/gemma3/all-data.csv'

df = pd.read_csv(
    file_path,
    names=["sentiment", "text"], # columns
    encoding='utf-8',
    encoding_errors="replace" # 오류가 있는 문자를 대체 문자로 바꿈
)

In [11]:
X_train, X_test = [], []
y_true_list = []

In [12]:
df.head()

,sentiment,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


In [13]:
df[df.sentiment == 'positive']

,sentiment,text
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...
5,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...
6,positive,"For the last quarter of 2010 , Componenta 's n..."
7,positive,"In the third quarter of 2010 , net sales incre..."
...,...,...
4775,positive,"The apartment block will be well-located , in ..."
4780,positive,"The antibody , given at repeated doses of up t..."
4786,positive,Danske Bank A-S DANSKE DC jumped 3.7 percent t...
4787,positive,Our superior customer centricity and expertise...


In [18]:
df[df.sentiment == 'positive']["sentiment"]

3       positive
4       positive
5       positive
6       positive
7       positive
          ...   
4775    positive
4780    positive
4786    positive
4787    positive
4822    positive
Name: sentiment, Length: 1363, dtype: object

In [23]:
train, test = train_test_split(df[df.sentiment == 'positive'],
                               train_size=300,
                               test_size=300,
                               random_state=42,
                                shuffle=True)

In [24]:
len(train), len(test)

(300, 300)

In [25]:
X_train_parts, X_test_parts = [], []

In [ ]:
for sentiment, group in df.groupby("sentiment"):
    train, test = train_test_split(
        group,
        train_size=300,
        test_size=300,
        random_state=42,
        shuffle=True
    )

    X_train_parts.append(train) # 3가지 모두 하나에 합침
    X_test_parts.append(test)
    

In [44]:
X_train = pd.concat(X_train_parts).sample(frac=1, random_state=10)

In [45]:
X_train

,sentiment,text
3683,neutral,Mr Jortikka is president of the base metal div...
4800,negative,`` Operating profit declined mainly due to the...
2250,positive,"Under the agreement , TietoEnator will provide..."
4055,negative,Finnish forest machinery manufacturer Ponsse h...
4071,negative,Scanfil has also issued a profit warning .
...,...,...
1374,neutral,"The dividend will be paid on April 15 , 2008 t..."
3869,neutral,The new shares entitle their holders to divide...
2766,neutral,Activities range from the development of natur...
3987,negative,The personnel reductions will primarily affect...


In [46]:
X_test_full = pd.concat(X_test_parts)

In [47]:
y_true = X_test_full["sentiment"]
X_test = X_test_full[["text"]]

In [48]:
y_true, X_test

(3790    negative
 4670    negative
 4797    negative
 2743    negative
 4065    negative
           ...   
 239     positive
 385     positive
 165     positive
 2110    positive
 791     positive
 Name: sentiment, Length: 900, dtype: object,
                                                    text
 3790  The company decided at the end of 2008 to temp...
 4670  down to EUR5 .9 m H1 '09 3 August 2009 - Finni...
 4797  The steelmaker said that the drop in profit wa...
 2743  Finland-based Stockmann Group has closed seven...
 4065  Operating loss before non-recurring items was ...
 ...                                                 ...
 239   Nokia Multimedia 's net sales totaled EUR 5.7 ...
 385   The total delivery volume of paper businesses ...
 165   Both operating profit and net sales for the ni...
 2110  For Telenor , the three and a half year contra...
 791   The rebuilds are designed to improve the machi...
 
 [900 rows x 1 columns])

In [49]:
# Prepare Eval Data
train_indices = set(X_train.index)
test_indices = set(X_test_full.index)

In [50]:
test_indices == train_indices

False

In [53]:
selected_indices = train_indices | test_indices

In [54]:
# train, test 모두 해당되지 않는걸 eval로 설정
X_eval = df.loc[~df.index.isin(selected_indices)].copy()

In [55]:
X_eval

,sentiment,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...
7,positive,"In the third quarter of 2010 , net sales incre..."
...,...,...
4807,neutral,The winner does not have to be present to win .
4809,neutral,Pentik+�inen emphasises that the most of the i...
4812,neutral,Because expenditures must be justified to pass...
4819,neutral,"Nevertheless , the development can not be allo..."


In [ ]:
X_eval = X_eval.groupby(
    'sentiment',
    group_keys=False # 그룹을 하나로 합침. 이때 원래는 새로운 인덱스를 넣는데 MultiIndex가 되버려서 이걸 방지
).apply( # 그룹마다 적용할 함수
    lambda x: x.sample(n=50, random_state=10, replace=True) # 해당 그룹에서 무작위로 50개 선택
).reset_index(drop=True)

/tmp/ipykernel_3314642/1474748762.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply( # 그룹마다 적용할 함수


In [59]:
X_eval

,sentiment,text
0,negative,"In addition , the company will reduce a maximu..."
1,negative,"In addition , the company will reduce a maximu..."
2,negative,A total of 140 jobs will be reduced at the Raa...
3,negative,Finnish GeoSentric 's net sales decreased to E...
4,negative,A total of 140 jobs will be reduced at the Raa...
...,...,...
145,positive,The three year turn-around program is expected...
146,positive,Diluted EPS rose to EUR3 .68 from EUR0 .50 .
147,positive,"At the end of March 2007 , the group 's order ..."
148,positive,Operating profit improved by 16.7 % to EUR 7.7...


In [70]:
# -- Prompt Generation Functions --

# Function to generate training prompts (with label)
def generate_train_prompt(data_point):
    return f"""
    Analyze the sentiment of the news headline enclosed in square brackets.
    Determine if it is positive, neutral, or negative, and return the corresponding sentiment label: "positive", "neutral", or "negative".

    [{data_point["text"]}] = {data_point["sentiment"]}
    """.strip() + EOS_TOKEN # Add EOS token

def generate_test_prompt(data_point):
    return f"""
    Analyze the sentiment of the news headline enclosed in square brackets.
    Determine if it is positive, neutral, or negative, and return the corresponding sentiment label: "positive", "neutral", or "negative".

    [{data_point["text"]}] = """.strip() # No label or EOS token needed here for generation

In [71]:
X_train = pd.DataFrame(X_train.apply(generate_train_prompt, axis=1), columns=["text"])
X_eval = pd.DataFrame(X_eval.apply(generate_train_prompt, axis=1), columns=["text"])
X_test = pd.DataFrame(X_test.apply(generate_test_prompt, axis=1), columns=["text"])


In [75]:
X_train

,text
3683,Analyze the sentiment of the news headline enc...
4800,Analyze the sentiment of the news headline enc...
2250,Analyze the sentiment of the news headline enc...
4055,Analyze the sentiment of the news headline enc...
4071,Analyze the sentiment of the news headline enc...
...,...
1374,Analyze the sentiment of the news headline enc...
3869,Analyze the sentiment of the news headline enc...
2766,Analyze the sentiment of the news headline enc...
3987,Analyze the sentiment of the news headline enc...


In [84]:
# Convert pandas DF to HF Dataset
train_data = Dataset.from_pandas(X_train, preserve_index=False)
eval_data = Dataset.from_pandas(X_eval, preserve_index=False)

In [85]:
train_data

Dataset({
    features: ['text'],
    num_rows: 900
})

In [86]:
train_data[0]

{'text': 'Analyze the sentiment of the news headline enclosed in square brackets.\n    Determine if it is positive, neutral, or negative, and return the corresponding sentiment label: "positive", "neutral", or "negative".\n\n    [Mr Jortikka is president of the base metal division of Outotec Oyj in Finland and is on the executive committee of Outotec .] = neutral<eos>'}

In [87]:
eval_data

Dataset({
    features: ['text'],
    num_rows: 150
})

In [ ]:
def evaluate(y_true, y_pred):
    """
    Evaluate the fine-tuned sentiment model's performance
    """

    # Define sentiment label mapping to numeric for scikit-learn metrics
    label_mapping = {'positive': 2, 'neutral': 1, 'negative': 0}

    # Handle 'none' predictions (map to neutral, or handle as error if preferred)
    y_true_num = np.array([label_mapping.get(label, 1) for label in y_true])
    y_pred_num = np.array([label_mapping.get(label, 1) for label in y_pred])

    # Calculate overall accuracy
    accuracy = accuracy_score(y_true_num, y_pred_num)
    print(f'Overall Accuracy: {accuracy:.3f}')

    # Compute accuracy for each sentiment label
    # 유니크한 np.unique([1,2,3,3,3,2,5]) = [1,2,3,5]
    unique_labels = np.unique(y_true_num) # Get unique numeric labels

    # Map numeric back to string for printing
    reverse_label_mapping = {v: k for k, v in label_mapping.items()}


    # 라벨별 accuracy를 계산하려고 함
    for label_num in unique_labels:
        # 현재 라벨에 해당하는 마스크만
        label_mask = y_true_num == label_num # Mask for current class
        label_accuracy = accuracy_score(y_true_num[label_mask], y_pred_num[label_mask])
        print(f'Accuracy for label {label_num} ({reverse_label_mapping.get(label_num, "unknown")}): {label_accuracy:.3f}')

    # Generate classification report using string labels for clarity
    class_report = classification_report(y_true, y_pred, labels=["negative", "neutral", "positive"], zero_division=0)
    print('\nClassification Report:\n', class_report)

    # Compute and display confusion matrix (using numeric labels)
    # Ensure labels are ordered correctly: negative(0), neutral(1), positive(2)
    conf_matrix = confusion_matrix(y_true_num, y_pred_num, labels=[0, 1, 2])
    print('\nConfusion Matrix (Rows: True, Cols: Pred) [Neg, Neu, Pos]:\n', conf_matrix)

In [91]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((11

In [ ]:
y_pred = []
model.eval()

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((11

In [96]:
prompt = X_test.iloc[0]["text"]
prompt

'Analyze the sentiment of the news headline enclosed in square brackets.\n    Determine if it is positive, neutral, or negative, and return the corresponding sentiment label: "positive", "neutral", or "negative".\n\n    [The company decided at the end of 2008 to temporarily shut down its ammonia plant in Billingham and extend the maintenance period at its Ince facility .] ='

In [111]:
encoded = tokenizer(prompt, return_tensors="pt").to(device)
encoded

{'input_ids': tensor([[     2, 115863,    506,  27078,    529,    506,   4668,  50196,  35585,
            528,   6281,  41706, 236761,    107,    140, 102752,    768,    625,
            563,   4414, 236764,  12643, 236764,    653,   5676, 236764,    532,
            994,    506,   7041,  27078,   5346, 236787,    623,  30558,    827,
            623,  45258,    827,    653,    623,  27851,   3056,    108,    140,
         236840,    818,   2544,   6544,    657,    506,   1345,    529, 236743,
         236778, 236771, 236771, 236828,    531,  32832,  13213,   1679,   1061,
          39612,   3732,    528, 117229,   3700,    532,  12975,    506,   9626,
           2846,    657,   1061,    799,    588,  10056,    783, 236842,    578]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [112]:
with torch.no_grad():
    outputs = model.generate(**encoded, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)

/opt/conda/lib/python3.11/site-packages/torch/_inductor/compile_fx.py:236: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


In [113]:
outputs

tensor([[     2, 115863,    506,  27078,    529,    506,   4668,  50196,  35585,
            528,   6281,  41706, 236761,    107,    140, 102752,    768,    625,
            563,   4414, 236764,  12643, 236764,    653,   5676, 236764,    532,
            994,    506,   7041,  27078,   5346, 236787,    623,  30558,    827,
            623,  45258,    827,    653,    623,  27851,   3056,    108,    140,
         236840,    818,   2544,   6544,    657,    506,   1345,    529, 236743,
         236778, 236771, 236771, 236828,    531,  32832,  13213,   1679,   1061,
          39612,   3732,    528, 117229,   3700,    532,  12975,    506,   9626,
           2846,    657,   1061,    799,    588,  10056,    783, 236842,    578,
            108, 203481, 236787,   5676,    108]], device='cuda:0')

In [114]:
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

In [115]:
decoded

'Analyze the sentiment of the news headline enclosed in square brackets.\n    Determine if it is positive, neutral, or negative, and return the corresponding sentiment label: "positive", "neutral", or "negative".\n\n    [The company decided at the end of 2008 to temporarily shut down its ammonia plant in Billingham and extend the maintenance period at its Ince facility .] =\n\nSentiment: negative\n\n'

In [116]:
prompt_end_marker = "] ="
generated_txt = decoded.split(prompt_end_marker)[1].strip().lower()

In [117]:
generated_txt

'sentiment: negative'

In [ ]:
def predict(
        X_test_df, 
        model_to_use,
        tokenizer_to_use,
        device_to_use=device,
        max_new_tokens=5, 
        temperature=0.01
):
    y_pred = [] # List to store predicted sentiment labels
    model_to_use.eval() # Set model to evaluation mode

    for i in tqdm(range(len(X_test_df)), desc="Predicting Sentiments"):
        prompt = X_test_df.iloc[i]['text']
        
        # Tokenize the prompt and move tensors to the correct device.
        encoded = tokenizer_to_use(prompt, return_tensors='pt').to(device_to_use)

        # Generate output from the model
        with torch.no_grad():
            outputs = model_to_use.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                pad_token_id=tokenizer_to_use.eos_token_id  # Avoid warning ID를 넣어야함 str token이 아니라.
            )
        
        # Decode the generated tokens (excluding the input prompt)
        # Find the start of the generated part by looking after the prompt structure
        prompt_end_marker = "]= " # 헤드라인 바로 뒤쪽
        decoded = tokenizer_to_use.decode(outputs[0], skip_special_tokens=True)

        # Extract only the generated part after the prompt marker
        try:
            generated_text = decoded.split(prompt_end_marker)[1].strip().lower()
        except IndexError:
            generated_text = ""

        if "positive" in generated_text:
            y_pred.append("positive")
        elif "negative" in generated_text:
            y_pred.append("negative")
        elif "neutral" in generated_text:
            y_pred.append("neutral")
        else:
            # Fallback if no clear label is found in the short generation
            y_pred.append("none")
            # print(f"Warning: Could not parse sentiment from: '{generated_text}' derived from '{full_decoded_text}'")

    return y_pred

In [ ]:
y_pred_base = predict(X_test, model, tokenizer)

Predicting Sentiments:   1%|          | 6/900 [00:56<1:34:21,  6.33s/it]Exception ignored in: <function WeakIdKeyDictionary.__init__.<locals>.remove at 0x7f4bbde79080>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/weak.py", line 125, in remove
    def remove(k, selfref=ref(self)):

KeyboardInterrupt: 
Predicting Sentiments:   1%|          | 10/900 [01:30<1:32:38,  6.25s/it]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f4f49444bd0>>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
# Evaluate the baseline predictions
print("--- Baseline Model Evaluation ---")
evaluate(y_true, y_pred_base)